In [2]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

# ==========================================
# 1. CONFIGURAZIONE AMBIENTE
# ==========================================
DATASET_PATH = "/home/stefano-u/fuzzing_lab/shared_corpus/dataset_final_qwen.jsonl"
OUTPUT_DIR = "/home/stefano-u/fuzzing_ml_env/modello_ebpf_sft_fase1"
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B"

print(f"[*] Inizializzazione SFT per Fuzzer eBPF su Laptop (8GB VRAM)")

# ==========================================
# 2. CARICAMENTO TOKENIZER
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ==========================================
# 3. PREPARAZIONE DATASET E SPLIT
# ==========================================
# Carichiamo il dataset
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

# Dividiamo il dataset (90% training, 10% validation)
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
train_data = dataset_split["train"]
val_data = dataset_split["test"]

def applica_prompt_engineering(sample):
    kernel_ver = sample.get("kernel_version", "unknown")
    bytecode = sample.get("bytecode_hex", "")
    
    if sample.get("is_valid", False):
        prompt = f"Kernel: {kernel_ver} | Status: VALID\n### BYTECODE:\n{bytecode}{tokenizer.eos_token}"
    else:
        err_reason = sample.get("error_reason_clean", "Unknown error")
        err_line = sample.get("error_line", -1)
        prompt = f"Kernel: {kernel_ver} | Status: INVALID | Error: {err_reason} | Error Line: {err_line}\n### BYTECODE:\n{bytecode}{tokenizer.eos_token}"
        
    return {"formatted_prompt": prompt}

def tokenizza_dati(sample):
    tokenized = tokenizer(
        sample["formatted_prompt"],
        max_length=768, 
        truncation=True,
        padding="max_length"
    )
    
    input_ids = tokenized["input_ids"]
    # Mascheriamo il padding per la Loss (-100 viene ignorato da PyTorch)
    labels = [
        (l if l != tokenizer.pad_token_id else -100) for l in input_ids
    ]
    
    return {
        "input_ids": input_ids,
        "attention_mask": tokenized["attention_mask"],
        "labels": labels
    }

print("[*] Processing training e validation dataset...")
# Applichiamo map() separatamente ai due split
train_data_preparato = train_data.map(applica_prompt_engineering).map(tokenizza_dati)
val_data_preparato = val_data.map(applica_prompt_engineering).map(tokenizza_dati)

# ==========================================
# 4. CARICAMENTO MODELLO (OTTIMIZZATO 4-BIT + BF16 + FLASH ATTENTION 2)
# ==========================================
print(f"[*] Caricamento {MODEL_NAME} in 4-bit (QLoRA) con FlashAttention-2 e BF16...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # <-- Cambiato in bfloat16
)

modello = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa"
)

modello.gradient_checkpointing_enable()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"] 
)

modello_efficiente = get_peft_model(modello, lora_config)
modello_efficiente.print_trainable_parameters()

# ==========================================
# 5. ADDESTRAMENTO CON VALIDAZIONE
# ==========================================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=1500, 
    per_device_train_batch_size=1, 
    gradient_accumulation_steps=8, 
    learning_rate=2e-4,
    bf16=True, # <-- Cambiato da fp16 a bf16
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_steps=100, # <-- Sincronizzato con gli eval_steps
    eval_strategy="steps", # <-- Valutazione attiva
    eval_steps=100,        # <-- Valuta ogni 100 step
    load_best_model_at_end=True, # <-- Carica il modello con la validation loss minore alla fine
    report_to="none"
)

trainer = Trainer(
    model=modello_efficiente,
    args=training_args,
    train_dataset=train_data_preparato,
    eval_dataset=val_data_preparato, # <-- Passato il validation set
)

print("\n[!!!] START TRAINING [!!!]\n")
trainer.train()

# ==========================================
# 6. SALVATAGGIO
# ==========================================
percorso_salvataggio = f"{OUTPUT_DIR}/adattatore_ebpf_v1"
trainer.save_model(percorso_salvataggio)
tokenizer.save_pretrained(percorso_salvataggio)
print(f"[+] Addestramento completato. Il miglior modello validato è stato salvato in: {percorso_salvataggio}")

[*] Inizializzazione SFT per Fuzzer eBPF su Laptop (8GB VRAM)


[*] Processing training e validation dataset...
[*] Caricamento Qwen/Qwen2.5-Coder-1.5B in 4-bit (QLoRA) con FlashAttention-2 e BF16...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815

[!!!] START TRAINING [!!!]



Step,Training Loss,Validation Loss
100,1.250698,1.256933
200,1.246363,1.241482
300,1.219965,1.236746
400,1.216408,1.235473
500,1.243628,1.234250
600,1.224151,1.232343
700,1.216439,1.231706
800,1.236404,1.231711
900,1.223665,1.231285
1000,1.240874,1.230586


[+] Addestramento completato. Il miglior modello validato è stato salvato in: /home/stefano-u/fuzzing_ml_env/modello_ebpf_sft_fase1/adattatore_ebpf_v1


In [2]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

# ==========================================
# 1. CONFIGURAZIONE AMBIENTE
# ==========================================
DATASET_PATH = "/home/stefano-u/fuzzing_lab/shared_corpus/dataset_final_qwen.jsonl"
OUTPUT_DIR = "/home/stefano-u/fuzzing_ml_env/modello_ebpf_sft_fase1_ass"
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B"

print(f"[*] Inizializzazione SFT per Fuzzer eBPF su Laptop (8GB VRAM)")

# ==========================================
# 2. CARICAMENTO TOKENIZER
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ==========================================
# 3. PREPARAZIONE DATASET E SPLIT
# ==========================================
# Carichiamo il dataset
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

# Dividiamo il dataset (90% training, 10% validation)
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
train_data = dataset_split["train"]
val_data = dataset_split["test"]

def applica_prompt_engineering(sample):
    kernel_ver = sample.get("kernel_version", "unknown")
    # Prendiamo il verifier_log invece dell'hex!
    assembly = sample.get("verifier_log", "") 
    
    if sample.get("is_valid", False):
        prompt = f"Kernel: {kernel_ver} | Status: VALID\n### ASSEMBLY:\n{assembly}{tokenizer.eos_token}"
    else:
        err_reason = sample.get("error_reason_clean", "Unknown error")
        prompt = f"Kernel: {kernel_ver} | Status: INVALID | Error: {err_reason}\n### ASSEMBLY:\n{assembly}{tokenizer.eos_token}"
        
    return {"formatted_prompt": prompt}

def tokenizza_dati(sample):
    tokenized = tokenizer(
        sample["formatted_prompt"],
        max_length=768, 
        truncation=True,
        padding="max_length"
    )
    
    input_ids = tokenized["input_ids"]
    # Mascheriamo il padding per la Loss (-100 viene ignorato da PyTorch)
    labels = [
        (l if l != tokenizer.pad_token_id else -100) for l in input_ids
    ]
    
    return {
        "input_ids": input_ids,
        "attention_mask": tokenized["attention_mask"],
        "labels": labels
    }

print("[*] Processing training e validation dataset...")
# Applichiamo map() separatamente ai due split
train_data_preparato = train_data.map(applica_prompt_engineering).map(tokenizza_dati)
val_data_preparato = val_data.map(applica_prompt_engineering).map(tokenizza_dati)

# ==========================================
# 4. CARICAMENTO MODELLO (OTTIMIZZATO 4-BIT + BF16 + FLASH ATTENTION 2)
# ==========================================
print(f"[*] Caricamento {MODEL_NAME} in 4-bit (QLoRA) con FlashAttention-2 e BF16...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # <-- Cambiato in bfloat16
)

modello = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa"
)

modello.gradient_checkpointing_enable()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"] 
)

modello_efficiente = get_peft_model(modello, lora_config)
modello_efficiente.print_trainable_parameters()

# ==========================================
# 5. ADDESTRAMENTO CON VALIDAZIONE
# ==========================================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=1500, 
    per_device_train_batch_size=1, 
    gradient_accumulation_steps=8, 
    learning_rate=2e-4,
    bf16=True, # <-- Cambiato da fp16 a bf16
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_steps=100, # <-- Sincronizzato con gli eval_steps
    eval_strategy="steps", # <-- Valutazione attiva
    eval_steps=100,        # <-- Valuta ogni 100 step
    load_best_model_at_end=True, # <-- Carica il modello con la validation loss minore alla fine
    report_to="none"
)

trainer = Trainer(
    model=modello_efficiente,
    args=training_args,
    train_dataset=train_data_preparato,
    eval_dataset=val_data_preparato, # <-- Passato il validation set
)

print("\n[!!!] START TRAINING [!!!]\n")
trainer.train()

# ==========================================
# 6. SALVATAGGIO
# ==========================================
percorso_salvataggio = f"{OUTPUT_DIR}/adattatore_ebpf_v1"
trainer.save_model(percorso_salvataggio)
tokenizer.save_pretrained(percorso_salvataggio)
print(f"[+] Addestramento completato. Il miglior modello validato è stato salvato in: {percorso_salvataggio}")

[*] Inizializzazione SFT per Fuzzer eBPF su Laptop (8GB VRAM)


[*] Processing training e validation dataset...


Map:   0%|          | 0/24762 [00:00<?, ? examples/s]

Map:   0%|          | 0/24762 [00:00<?, ? examples/s]

Map:   0%|          | 0/2752 [00:00<?, ? examples/s]

...

[*] Caricamento Qwen/Qwen2.5-Coder-1.5B in 4-bit (QLoRA) con FlashAttention-2 e BF16...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815

[!!!] START TRAINING [!!!]



Step,Training Loss,Validation Loss
100,0.708511,0.691678
200,0.684019,0.663427
300,0.640620,0.654140
400,0.649504,0.647003
500,0.641350,0.641362
600,0.634620,0.638821
700,0.635961,0.634541
800,0.648361,0.632460
900,0.624267,0.630197
1000,0.624508,0.628141


[+] Addestramento completato. Il miglior modello validato è stato salvato in: /home/stefano-u/fuzzing_ml_env/modello_ebpf_sft_fase1_ass/adattatore_ebpf_v1


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B"
# Assicurati che questo sia il percorso del modello addestrato sull'ASSEMBLY
ADAPTER_PATH = "/home/stefano-u/fuzzing_ml_env/modello_ebpf_sft_fase1_ass/adattatore_ebpf_v1"
OUTPUT_FILE = "generato.s"

print("[*] Caricamento modello...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

# Chiediamo al modello di generare un codice VALIDO
prompt_input = "Kernel: unknown | Status: VALID\n### ASSEMBLY:\n"

inputs = tokenizer(prompt_input, return_tensors="pt").to("cuda")

print("[*] Generazione in corso...")
outputs = model.generate(
    **inputs, 
    max_new_tokens=300, 
    temperature=0.7, # Un po' di creatività
    do_sample=True,
    eos_token_id=tokenizer.eos_token_id
)

testo_generato = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Estraiamo solo la parte dopo "### ASSEMBLY:\n"
assembly_puro = testo_generato.split("### ASSEMBLY:\n")[-1].strip()

# Salviamo il file con estensione .s (estensione standard per l'assembly)
with open(OUTPUT_FILE, "w") as f:
    f.write(assembly_puro + "\n")

print(f"\n[+] Programma generato e salvato in {OUTPUT_FILE}:")
print("--------------------------------------------------")
print(assembly_puro)
print("--------------------------------------------------")

[*] Caricamento modello...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


[*] Generazione in corso...

[+] Programma generato e salvato in generato.s:
--------------------------------------------------
func#0 @0
0: (b7) r0 = -164227997               ; r0 = 0xfffffffff84ec22d
1: (b7) r1 = -1141780649              ; r1 = 0xd76a9717
2: (b7) r2 = 1727591401               ; r2 = 0x67106819
3: (b7) r3 = 2115761146               ; r3 = 0x7e1c3b82
4: (b7) r4 = -633185296               ; r4 = 0xffffffffdb45b470
5: (b7) r5 = -142631611               ; r5 = 0xfffffffff74e419d
6: (b7) r6 = 1218509234               ; r6 = 0x48138276
7: (b7) r7 = 1616338337               ; r7 = 0x5f394091
8: (b
--------------------------------------------------


In [5]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from tqdm import tqdm

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B"
# Assicurati che questo sia il percorso del modello addestrato sull'ASSEMBLY
ADAPTER_PATH = "/home/stefano-u/fuzzing_ml_env/modello_ebpf_sft_fase1_ass/adattatore_ebpf_v1"
OUTPUT_FILE = "programmi_generati.json"
NUM_PROGRAMMI = 100

print("[*] Caricamento modello per Inferenza...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

prompt_input = "Kernel: unknown | Status: VALID\n### ASSEMBLY:\n"
inputs = tokenizer(prompt_input, return_tensors="pt").to("cuda")

programmi = []

print(f"[*] Generazione di {NUM_PROGRAMMI} programmi...")
for i in tqdm(range(NUM_PROGRAMMI)):
    outputs = model.generate(
        **inputs, 
        max_new_tokens=400, 
        temperature=0.8, # Un po' di "pazzia" per trovare bug
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )
    
    testo_generato = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Estraiamo solo il codice Verifier
    assembly = testo_generato.split("### ASSEMBLY:\n")[-1].strip()
    
    programmi.append({
        "id": i,
        "verifier_log": assembly
    })

with open(OUTPUT_FILE, "w") as f:
    json.dump(programmi, f, indent=4)

print(f"\n[+] Salvati {NUM_PROGRAMMI} programmi in '{OUTPUT_FILE}'")

[*] Caricamento modello per Inferenza...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[*] Generazione di 100 programmi...


100%|██████████████████████████████████████████████████████████████████████████████| 100/100 [3:39:15<00:00, 131.56s/it]


[+] Salvati 100 programmi in 'programmi_generati.json'


In [6]:
import json
import re
import struct
import os

INPUT_FILE = "programmi_generati.json"
OUT_DIR = "bytecode_compilati"

# Creiamo la cartella di output
os.makedirs(OUT_DIR, exist_ok=True)

def parse_verifier_line(line):
    """
    Analizza una riga del Verifier ed estrae i byte dell'istruzione eBPF.
    Istruzione eBPF (64 bit):
    8 bit opcode | 4 bit dst_reg | 4 bit src_reg | 16 bit offset | 32 bit immediate
    """
    # Cerchiamo il pattern "0: (b7) ...."
    match = re.match(r'^\d+:\s+\(([0-9a-fA-F]{2})\)\s+(.*)', line.strip())
    if not match:
        return None
    
    opcode = int(match.group(1), 16)
    rest = match.group(2)
    
    dst = 0
    src = 0
    off = 0
    imm = 0
    
    try:
        # 1. Operazioni con Immediato (es. r0 = -164227997)
        if re.match(r'^r(\d+)\s*[-+*/&|^]?=\s*(-?\d+)$', rest):
            m = re.match(r'^r(\d+)\s*[-+*/&|^]?=\s*(-?\d+)$', rest)
            dst = int(m.group(1))
            imm = int(m.group(2))
            
        # 2. Operazioni tra Registri (es. r1 = r2)
        elif re.match(r'^r(\d+)\s*[-+*/&|^]?=\s*r(\d+)$', rest):
            m = re.match(r'^r(\d+)\s*[-+*/&|^]?=\s*r(\d+)$', rest)
            dst = int(m.group(1))
            src = int(m.group(2))
            
        # 3. Memory Store (es. *(u64 *)(r10 - 8) = r1)
        elif re.search(r'\*\(.*?\)\s*\(r(\d+)\s*([+-]\s*\d+)?\)\s*=\s*r(\d+)', rest):
            m = re.search(r'\*\(.*?\)\s*\(r(\d+)\s*([+-]\s*\d+)?\)\s*=\s*r(\d+)', rest)
            dst = int(m.group(1))
            if m.group(2): off = int(m.group(2).replace(' ', ''))
            src = int(m.group(3))
            
        # 4. Memory Load (es. r1 = *(u64 *)(r10 - 8))
        elif re.search(r'^r(\d+)\s*=\s*\*\(.*?\)\s*\(r(\d+)\s*([+-]\s*\d+)?\)', rest):
            m = re.search(r'^r(\d+)\s*=\s*\*\(.*?\)\s*\(r(\d+)\s*([+-]\s*\d+)?\)', rest)
            dst = int(m.group(1))
            src = int(m.group(2))
            if m.group(3): off = int(m.group(3).replace(' ', ''))
            
        # 5. Call (es. call 5)
        elif rest.startswith("call"):
            m = re.search(r'call\s+(-?\d+)', rest)
            if m: imm = int(m.group(1))
            
        # 6. Exit (es. exit) 
        elif rest.startswith("exit"):
            pass # Tutto a 0 tranne l'opcode
            
    except ValueError:
        # Se i numeri sono fuori range per 32-bit (es. 0xfffffffff84ec22d)
        # Tronchiamo a 32-bit (come fa il kernel in C)
        if 'imm' in locals(): imm = imm & 0xFFFFFFFF
    
    # Mascheriamo a 32 bit per evitare errori di struct.pack con i numeri enormi negativi
    imm = imm & 0xFFFFFFFF
    if imm > 0x7FFFFFFF:
        imm -= 0x100000000 # Converti in signed 32-bit

    # Format eBPF: (src_reg << 4) | (dst_reg & 0x0f)
    regs = (src << 4) | (dst & 0x0f)
    
    # struct.pack '<BBhi':
    # < = Little Endian
    # B = unsigned char (1 byte) -> Opcode
    # B = unsigned char (1 byte) -> Registri
    # h = short (2 byte) -> Offset
    # i = int (4 byte) -> Immediato
    return struct.pack('<BBhi', opcode, regs, off, imm)


print(f"[*] Lettura da {INPUT_FILE}...")
with open(INPUT_FILE, "r") as f:
    programmi = json.load(f)

compilati_ok = 0

for prog in programmi:
    codice_sorgente = prog["verifier_log"]
    binary_payload = bytearray()
    
    for linea in codice_sorgente.split('\n'):
        if not linea.strip() or linea.startswith(";"): 
            continue
            
        byte_istruzione = parse_verifier_line(linea)
        if byte_istruzione:
            binary_payload.extend(byte_istruzione)
            
    # Salviamo solo se siamo riusciti ad assemblare qualcosa
    if len(binary_payload) > 0:
        nome_file = f"{OUT_DIR}/prog_{prog['id']}.bin"
        with open(nome_file, "wb") as f_out:
            f_out.write(binary_payload)
        compilati_ok += 1

print(f"\n[+] Compilazione completata!")
print(f"[+] Programmi assemblati in binario: {compilati_ok}/{len(programmi)}")
print(f"[+] I file .bin sono nella cartella '{OUT_DIR}/'.")

[*] Lettura da programmi_generati.json...

[+] Compilazione completata!
[+] Programmi assemblati in binario: 0/100
[+] I file .bin sono nella cartella 'bytecode_compilati/'.


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 1. Define the models
base_model_name_or_path = "Qwen/Qwen2.5-Coder-1.5B" 
adapter_path = "/home/stefano-u/fuzzing_ml_env/modello_ebpf_sft_fase1_ass/adattatore_ebpf_v1" # Make sure this is your absolute path

print("Loading base model and tokenizer...")

# 2. Load the base Tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name_or_path)

# 3. Load the Base Model 
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name_or_path,
    torch_dtype=torch.bfloat16,
    device_map={"": 0} # <--- THIS IS THE FIX
)

print("Applying your eBPF LoRA adapter...")

# 4. Snap your adapter onto the base model
model = PeftModel.from_pretrained(base_model, adapter_path)

print("Success! Ready for questions.")

Loading base model and tokenizer...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Applying your eBPF LoRA adapter...
Success! Ready for questions.


In [6]:
def ask_qwencoder(question, max_tokens=512, temp=0.7):
    # Format the prompt
    messages = [{"role": "user", "content": question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # Send text to model
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Generate answer
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=temp
    )
    
    # Strip the prompt out of the generated response
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

In [8]:
answer = ask_qwencoder("can u generate assembly instruction for and ebpf bytecode that is aimed to use the pointer arithmetic strategy")
print(answer)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Certainly! Below is an example of an assembly instruction sequence and an eBPF (Extended Berkeley Packet Filter) bytecode sequence that use the pointer arithmetic strategy. This example assumes a hypothetical architecture where registers are named `r0` through `r7` and stack pointers are named `sp`.

### Assembly Instruction Sequence
```assembly
// Assume we have a pointer `ptr` and we want to calculate `ptr + 100`
mov r0, sp  ; Save the current stack pointer
sub sp, 100  ; Adjust the stack pointer by -100
```

### eBPF Bytecode Sequence
```c
struct bpf_insn insns[] = {
    {BPF_MOV, 0, 0, 0},  // r0 = 0
    {BPF_MOV, 0, 1, 0},  // r1 = 1
    {BPF_MOV, 0, 2, 0},  // r2 = 2
    {BPF_MOV, 0, 3, 0},  // r3 = 3
    {BPF_MOV, 0, 4, 0},  // r4 = 4
    {BPF_MOV, 0, 5, 0},  // r5 = 5
    {BPF_MOV, 0, 6, 0},  // r6 = 6
    {BPF_MOV, 0, 7, 0},  // r7 = 7
    {BPF_MOV, 0, 8, 0},  // r8 = 8
    {BPF_MOV, 0, 9, 0},  // r9 = 9
    {BPF_MOV, 0, 10, 0}, // r10 = 10
    {BPF_MOV, 0, 11, 0}, // r11 = 11

In [9]:
def generate_ebpf_assembly(kernel="unknown", status="VALID", max_tokens=500, temp=0.7):
    """
    Generates a raw eBPF verifier assembly log using the fine-tuned LoRA adapter.
    """
    # 1. Create the EXACT trigger phrase the model expects from your training data
    prompt = f"Kernel: {kernel} | Status: {status}\n### ASSEMBLY:\n"
    
    # 2. Prepare the input for the GPU
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # 3. Generate the sequence
    outputs = model.generate(
        **inputs, 
        max_new_tokens=max_tokens, 
        temperature=temp,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )
    
    # 4. Decode the raw output
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # 5. Extract and clean up ONLY the assembly code
    assembly = generated_text.split("### ASSEMBLY:\n")[-1].strip()
    
    return assembly

In [10]:
# Just call the function with default settings
my_program = generate_ebpf_assembly()

print(my_program)

func#0 @0
0: (b7) r0 = -57335293               ; r0 = 0xfffffffffd334263
1: (b7) r1 = -78729408               ; r1 = 0xfffffffffb146c50
2: (b7) r2 = 1171597141              ; r2 = 0x4700a43d
3: (b7) r3 = 596065562               ; r3 = 0x237b413e
4: (b7) r4 = 1135368455              ; r4 = 0x442b308b
5: (b7) r5 = -1327913733             ; r5 = 0xffffffffb1183b25
6: (b7) r6 = 1076238891              ; r6 = 0x408477cf
7: (b7) r7 = -1706056068             ; r7 = 0xffffffff9b095074
8: (b7) r8 = -1545870193             ; r8 = 0xffffffffa403865f
9: (b7) r9 = -1726298319             ; r9 = 0xffffffff99e52241
10: (a5) if r0 < 0x393a950e goto pc+171
mark_precise: regs=r0 stack=<|fim_middle|>
118: (3c) w7 /= w5                    ; w5=0xffffffffb1183b25, w7=0x4700a43d
119: (5e) if w9 != w5 goto pc+191
mark_precise: regs=r9 stack=


In [13]:
import subprocess
import re
import tempfile
import os

def verifier_to_elf(assembly_text, output_elf_path="output_ebpf.o"):
    """
    Parses verifier assembly log strings, filters out incomplete/broken instructions,
    and compiles them into a loadable eBPF ELF object.
    """
    assembly_lines = assembly_text.split('\n')
    cleaned_instructions = []
    
    print("[*] Parsing verifier log into standard eBPF assembly...")
    
    # Matches a line number followed by parenthesis opcode, then captures the instruction
    # e.g., "10: (a5) if r0 < 0x393a950e goto pc+171" -> "if r0 < 0x393a950e goto pc+171"
    pattern = re.compile(r'^\s*\d+:\s*\([0-9a-fA-F]+\)\s*(.+?)(?:\s*;|$)')
    
    for line in assembly_lines:
        line = line.strip()
        # Skip metadata, helper lines, or model tags
        if not line or "mark_precise" in line or "<|" in line or line.startswith("func#"):
            continue
            
        match = pattern.match(line)
        inst = ""
        if match:
            inst = match.group(1).strip()
        elif "=" in line or "goto" in line or "exit" in line:
            # Fallback for instructions missing line numbers
            inst = line.split(';')[0].strip()

        # --- NEW: Robust validation checks ---
        if inst:
            # Reject incomplete lines like just "r4" or "if r0 <" 
            # Valid instructions usually contain an assignment (=) or a jump (goto/exit/call)
            has_assignment = "=" in inst
            has_jump = any(x in inst for x in ["goto", "exit", "call"])
            
            if has_assignment or has_jump:
                cleaned_instructions.append(inst)
            else:
                print(f"[!] Warning: Dropped incomplete instruction: '{inst}'")

    if not cleaned_instructions:
        print("[-] Error: No valid, complete instructions could be parsed from the text!")
        return False

    # Build the assembly source
    assembly_source = """
    .section socket
    .globl prog
prog:
"""
    for inst in cleaned_instructions:
        assembly_source += f"    {inst}\n"
    
    # Ensure a clean exit is appended so the program always terminates safely
    if "exit" not in [i.lower() for i in cleaned_instructions]:
        assembly_source += "    r0 = 0\n"
        assembly_source += "    exit\n"

    # Write assembly source and invoke Clang
    with tempfile.TemporaryDirectory() as tmpdir:
        asm_file = os.path.join(tmpdir, "temp.s")
        with open(asm_file, "w") as f:
            f.write(assembly_source)
            
        print("[*] Assembling code using Clang targeting eBPF...")
        try:
            cmd = [
                "clang", "-target", "bpf", 
                "-c", asm_file, 
                "-o", output_elf_path
            ]
            subprocess.run(cmd, check=True, capture_output=True, text=True)
            print(f"[+] Success! eBPF ELF file saved to: {output_elf_path}")
            return True
        except subprocess.CalledProcessError as e:
            print("[-] Clang Compilation Failed!")
            print(f"Error Details:\n{e.stderr}")
            return False

In [15]:
# 1. Generate the assembly using your fine-tuned model
raw_assembly = generate_ebpf_assembly(status="VALID", max_tokens=400)

print("--- Generated Assembly ---")
print(raw_assembly)
print("--------------------------")

# 2. Convert it into a loadable ELF binary!
success = verifier_to_elf(raw_assembly, output_elf_path="my_fuzz_program.o")

if success:
    print("[*] You can now load this file using bpftool or your fuzzer harness!")

--- Generated Assembly ---
func#0 @0
0: (b7) r0 = -1730293281              ; *memory_start_1=
1: (b7) r1 = -1956454644              ; *memory_start_2=
2: (b7) r2 = -1987750868              ; *memory_start_3=
3: (b7) r3 = 1862174139               ; *memory_start_4=
4: (b7) r4 = -1707671673              ; *memory_start_5=
5: (b7) r5 = -1577589385              ; *memory_start_6=
6: (b7) r6 = -2124126348              ; *memory_start_7=
7: (b7) r7 = 1992310679               ; *memory_start_8=
8: (b7) r8 = 1472294586               ; *memory_start_9=0x56086072
9: (b7) r9 = 1118746323               ; *memory_start_10=0x439d907b
10: (57) r8 &= 1781500494             ; *memory_start_10=0x43080072
11: (6c) w0 <<= w7                    ; *memory_start_1=0xdd0c4000
12: (54) w5 &= -512105369
--------------------------
[*] Parsing verifier log into standard eBPF assembly...
[*] Assembling code using Clang targeting eBPF...
[+] Success! eBPF ELF file saved to: my_fuzz_program.o
[*] You can now load th